# 04 — Comparison and pre-registered decisions

Paired match bootstrap (1000 draws, seed 20261108, 95 %). Primary: `L_ctrl`, `target_like`. Everything else is report-only (PREREG §6).

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp3_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_3')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp3_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp3_protocol as P, exp3_report as R
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PREREG verified cff444fff0106e9c | frozen 2026-11-08T16:53:50+00:00


In [2]:
parts = [pd.read_csv(C.RESULTS / 'reference_rows.csv'), pd.read_csv(C.RESULTS / 'argmax_rows.csv')]
for label_set in R.S1_LABELS:
    for variant in P.VARIANTS:
        for view in C.VIEWS:
            parts.append(pd.DataFrame(R.variant_rows(variant, view, label_set)))
rows = pd.concat(parts, ignore_index=True)
rows.to_csv(C.RESULTS / 'per_match_results.csv', index=False)
table = R.summary_table(rows)
table.to_csv(C.RESULTS / 'comparison_table.csv', index=False)
cols = ['variant', 'mean_spell_f1', 'min_spell_f1', 'mean_macro_f1', 'mean_abs_count_error', 'max_abs_count_error',
        'delta_vs_ref_ens3', 'ci95_low_vs_ref', 'ci95_high_vs_ref', 'wins_vs_ref', 'delta_vs_exp2_aug', 'ci95_low_vs_exp2', 'ci95_high_vs_exp2', 'count_gate']
for label_set in R.S1_LABELS:
    for view in C.VIEWS:
        part = table[(table.label_set == label_set) & (table.view == view)]
        print(f'== {label_set} / {view}'); print(R.as_markdown(part, cols)); print()

== ctrl / original
| variant | mean_spell_f1 | min_spell_f1 | mean_macro_f1 | mean_abs_count_error | max_abs_count_error | delta_vs_ref_ens3 | ci95_low_vs_ref | ci95_high_vs_ref | wins_vs_ref | delta_vs_exp2_aug | ci95_low_vs_exp2 | ci95_high_vs_exp2 | count_gate |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| exp3_ctrl_ens3 | 0.741 | 0.689 | 0.922 | 0.134 | 0.175 | -0.010 | -0.018 | +0.001 | 2 | +0.011 | -0.003 | +0.026 | True |
| exp3_ref_ens3 | 0.751 | 0.707 | 0.925 | 0.113 | 0.195 | +0.000 | +0.000 | +0.000 | 0 | +0.021 | +0.007 | +0.035 | True |
| exp3_ctrl | 0.728 | 0.661 | 0.920 | 0.154 | 0.183 | -0.022 | -0.031 | -0.015 | 0 | -0.002 | -0.021 | +0.015 | True |
| exp3_ctrl_s1 | 0.722 | 0.680 | 0.918 | 0.123 | 0.197 | -0.029 | -0.043 | -0.013 | 1 | -0.008 | -0.024 | +0.010 | True |
| exp3_ctrl_s2 | 0.730 | 0.689 | 0.917 | 0.124 | 0.190 | -0.021 | -0.035 | -0.007 | 1 | -0.000 | -0.016 | +0.016 | True |
| exp2_aug | 0.730 | 0.690 | 0.917 | 0.106 | 0.217 | -0.021 | -0.0

## exp_0 quality gates on `L_ctrl` (vs the same view's baseline)

In [3]:
gate_rows = R.gate_rows(rows)
print(pd.DataFrame([{'view': g['view'], 'variant': g['variant'], **g['checks'], 'pass': g['pass'],
                     'worst_count_error': round(g['worst_count_error'], 3), 'worst_match': g['worst_count_match']}
                    for g in gate_rows]).to_string(index=False))

       view               variant  gain_at_least_005  ci_lower_above_zero  macro_regression_at_most_001  all_match_counts_within_30pct  pass  worst_count_error worst_match
   original        exp3_ctrl_ens3               True                 True                          True                           True  True              0.175      J03WN1
   original         exp3_ref_ens3               True                 True                          True                           True  True              0.195      J03WR9
   original             exp3_ctrl               True                 True                          True                           True  True              0.183      J03WOH
   original          exp3_ctrl_s1               True                 True                          True                           True  True              0.197      J03WOH
   original          exp3_ctrl_s2               True                 True                          True                           True  True

## Pre-registered decision (H1) and the adoption tree

In [4]:
decision = R.decide(rows)
C1.write_json(C.RESULTS / 'decision.json', decision)
h1 = decision['rules']['H1']
print(f"H1 {'PASSED' if h1['passed'] else 'FAILED'}: exp3_ctrl_ens3 - exp3_ref_ens3 (L_ctrl, target_like) = {h1['mean_delta']:+.4f} [{h1['ci95'][0]:+.4f}, {h1['ci95'][1]:+.4f}], wins {h1['wins']}/7")
print('candidate:', decision['candidate_variant'], '| final config:', {k: round(v, 3) for k, v in decision['candidate_final_config'].items()})
ro = decision['report_only']
for item in [ro['H1_original_view']] + ro['ctrl_ens3_vs_exp2_aug'] + ro['ref_ens3_vs_exp2_aug'] + ro['S1_ctrl_ens3_vs_ref_ens3']:
    print(f"  {item['label_set']:8s} {item['view']:12s} {item['a']:15s} - {item['b']:14s} {item['mean_delta']:+.4f} [{item['ci95'][0]:+.4f}, {item['ci95'][1]:+.4f}] wins {item['wins']}")
print('seed spread (L_ctrl):', {k: round(v, 4) for k, v in ro['seed_spread'].items()})

H1 FAILED: exp3_ctrl_ens3 - exp3_ref_ens3 (L_ctrl, target_like) = -0.0065 [-0.0140, +0.0018], wins 2/7
candidate: exp3_ref_ens3 | final config: {'temperature': 1.17, 'lambda_team': 8.0, 'lambda_dead': 4.0, 'gamma': 0.0}
  ctrl     original     exp3_ctrl_ens3  - exp3_ref_ens3  -0.0097 [-0.0183, +0.0010] wins 2
  ctrl     original     exp3_ctrl_ens3  - exp2_aug       +0.0111 [-0.0033, +0.0257] wins 4
  ctrl     target_like  exp3_ctrl_ens3  - exp2_aug       +0.0072 [-0.0067, +0.0206] wins 5
  ctrl     original     exp3_ref_ens3   - exp2_aug       +0.0208 [+0.0066, +0.0354] wins 5
  ctrl     target_like  exp3_ref_ens3   - exp2_aug       +0.0137 [-0.0016, +0.0261] wins 6
  ctrl     original     exp3_ctrl_ens3  - exp3_ref_ens3  -0.0097 [-0.0183, +0.0010] wins 2
  ctrl     target_like  exp3_ctrl_ens3  - exp3_ref_ens3  -0.0065 [-0.0140, +0.0018] wins 2
  provider original     exp3_ctrl_ens3  - exp3_ref_ens3  -0.0708 [-0.0788, -0.0629] wins 0
  provider target_like  exp3_ctrl_ens3  - exp3_ref_e

## S2: coverage − target_like (paired by match)

In [5]:
coverage = pd.read_csv(C.RESULTS / 'coverage_rows.csv')
shift = R.coverage_shift(rows, coverage)
print(pd.DataFrame(shift).round(4).to_string(index=False))
report = {'prereg_sha256': protocol['prereg_sha256'], 'bootstrap': 'paired match bootstrap, 1000 draws, seed 20261108, 95% percentile',
          'table': R.records(table), 'gates_ctrl': gate_rows, 'decision': decision, 'coverage_shift': shift,
          'coverage_stats': pd.read_csv(C.RESULTS / 'coverage_stats.csv').to_dict(orient='records')}
C1.write_json(C.RESULTS / 'proxy_report.json', report)

       variant label_set  mean_change                                           ci95  wins  mean_count_error_change
      exp2_aug      ctrl       0.0119    [0.00433796396289633, 0.019127657359497976]     6                   0.0148
      exp2_aug  provider       0.0128   [0.005777072666024373, 0.017869822563075947]     6                   0.0088
     exp3_ctrl      ctrl       0.0165    [0.00958583516995265, 0.024071996407652813]     6                   0.0070
     exp3_ctrl  provider       0.0079  [0.0014463509342195876, 0.013747241300849547]     5                  -0.0072
exp3_ctrl_ens3      ctrl       0.0123    [0.00730659262918365, 0.017304535920854534]     7                   0.0046
exp3_ctrl_ens3  provider       0.0096   [0.003614136393603156, 0.015569988173125715]     6                   0.0004
 exp3_ref_ens3      ctrl       0.0069 [-2.313333410914974e-05, 0.013676893687874373]     5                   0.0145
 exp3_ref_ens3  provider       0.0095   [0.0022552383822222973, 0.016839

## Figure (proxy only)

In [6]:
order = ['baseline', 'ryu_final', 'exp1_ref', 'exp2_aug', 'exp3_ref_ens3', 'exp3_ctrl', 'exp3_ctrl_ens3']
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for label_set, marker in zip(R.S1_LABELS, ('o', 's', '^', 'v')):
    part = table[(table.label_set == label_set) & (table.view == 'target_like')].set_index('variant').reindex(order)
    axes[0].plot(order, part.mean_spell_f1, marker=marker, label=label_set)
    axes[1].plot(order, part.mean_count_error, marker=marker, label=label_set)
axes[0].set_ylabel('mean spellF1, target_like (7 held-out matches)'); axes[0].set_title('spellF1 by arm and evaluation label set')
axes[1].axhline(0, color='0.5', lw=0.8); axes[1].set_ylabel('mean count error (pred / true - 1)'); axes[1].set_title('count bias')
for ax in axes:
    ax.tick_params(axis='x', rotation=35); ax.legend(title='labels')
fig.tight_layout(); fig.savefig(C.RESULTS / 'fig_exp3_labels.png', dpi=130)
print('saved', C.RESULTS / 'fig_exp3_labels.png')

saved /home/jeremy-mboe/Documents/Kuliah/Sem5/barca/exp_jeremy/exp_3/results/fig_exp3_labels.png
